# Lab 4 - Genie Agent for Aircraft Analytics

Lab 4 builds a **Genie Agent** over the aircraft sensor telemetry in the Lakehouse. Ask it a question in plain English and it writes the SQL, runs it, and hands back the answer. No SQL from you.

The Genie Agent is not a dead end. **Lab 5** uses it as one of the three tools its LangGraph supervisor routes across, so what you build here you use again.

**Time:** about 30 minutes.

## Prerequisites

- A Databricks workspace with Genie access
- A Serverless SQL Warehouse you can select
- The workshop catalog `databricks-neo4j-workshop` loaded by your admin

**Recommended:** finish **Lab 2** first. This lab queries shared Lakehouse tables rather than your own Aura instance, but Lab 2 teaches you the data model: aircraft topology, sensor relationships, flights, and maintenance events. Labs 5 and 6 do need your own loaded instance.

## Optional demo

[`OPTIONAL_COMPOUND_AGENT_DEMO.md`](./OPTIONAL_COMPOUND_AGENT_DEMO.md) takes this Genie Agent and makes it one half of a **compound AI agent**: a Supervisor Agent above it, and a second agent querying Neo4j over MCP. It is optional because it needs a hosted MCP server, which is outside the scope of the workshop. The instructor demos it.

---

# Genie Agent for Aircraft Sensor Analytics

You build a Genie Agent that answers natural language questions over aircraft sensor telemetry. It becomes one of the three tools in Lab 5.

## Step 1: Explore the Lakehouse Data

Your workshop admin pre-loaded a set of tables into Unity Catalog holding the Aircraft Digital Twin sensor telemetry. This is the data you point Genie at, so look at it first.

You can browse it in the UI:

1. Click **Catalog** in the left sidebar.
2. Expand **databricks-neo4j-workshop > aircraft**.
3. Click a table such as `sensor_readings` and open the **Sample Data** tab.

![Lakehouse sensor_readings table in Unity Catalog](https://raw.githubusercontent.com/neo4j-partners/databricks-neo4j-workshop/main/site/modules/ROOT/images/lab4-lakehouse-sensor-readings.png)

The four tables Genie needs:

| Table | Description |
|-------|-------------|
| `aircraft` | Fleet inventory: tail numbers, models, manufacturers, operators |
| `systems` | Aircraft systems: engines, avionics, hydraulics |
| `sensors` | Sensor metadata: EGT (Exhaust Gas Temperature), vibration, N1 speed (fan speed), fuel flow |
| `sensor_readings` | Telemetry every 4 hours over 90 days, July to September 2024 |

They form a join chain, `sensor_readings` to `sensors` to `systems` to `aircraft`, that connects a raw telemetry value all the way up to fleet metadata. Genie walks that chain to answer questions.

`SHOW TABLES` in the next cell lists eight, not four. The other four, `flights`, `maintenance_events`, `fleet_readiness` and `sensor_health`, are published by the same pipeline but stay out of this agent. Four tables is a small enough schema that Genie picks the right joins nearly every time, and the four extra ones invite the questions Neo4j answers better.

The [workshop glossary](https://neo4j-partners.github.io/databricks-neo4j-workshop/databricks-neo4j-workshop/1.0/glossary.html) expands EGT, N1 and the rest of the sensor vocabulary. Step 3 repeats those definitions inside the instructions block, because Genie needs them too.

The cells below query the same tables from this notebook. Run them.

In [ ]:
# The workshop catalog and schema. Both names come from lab/workshop.py,
# the one definition of every object this course creates.
CATALOG = "databricks-neo4j-workshop"
SCHEMA = "aircraft"

# The catalog name contains hyphens, so it has to be backticked in SQL.
FQN = f"`{CATALOG}`.`{SCHEMA}`"

spark.sql(f"USE CATALOG `{CATALOG}`")
spark.sql(f"USE SCHEMA `{SCHEMA}`")

print(f"Reading from {FQN}")
display(spark.sql(f"SHOW TABLES IN {FQN}"))

### The telemetry itself

`sensor_readings` is the fact table: one row per sensor per timestamp, 155,520 rows in total.

In [ ]:
display(spark.sql(f"SELECT * FROM {FQN}.sensor_readings LIMIT 20"))

In [ ]:
# How much data, over what window, across how many sensors.
display(
    spark.sql(f"""
        SELECT
            COUNT(*)                  AS reading_count,
            COUNT(DISTINCT sensor_id) AS sensor_count,
            MIN(timestamp)            AS first_reading,
            MAX(timestamp)            AS last_reading
        FROM {FQN}.sensor_readings
    """)
)

### The dimensions around it

`sensors` says what each reading measures and in what unit. `systems` and `aircraft` say what it is bolted to.

In [ ]:
display(
    spark.sql(f"""
        SELECT type, unit, COUNT(*) AS sensor_count
        FROM {FQN}.sensors
        GROUP BY type, unit
        ORDER BY type
    """)
)

In [ ]:
display(
    spark.sql(f"""
        SELECT manufacturer, model, operator, COUNT(*) AS aircraft_count
        FROM {FQN}.aircraft
        GROUP BY manufacturer, model, operator
        ORDER BY manufacturer, model
    """)
)

### The join chain, walked once by hand

This is the query shape Genie generates for you. Average EGT per model, reading telemetry values and grouping them by fleet metadata three joins away.

Note the spread across models. A fleet-wide EGT average mixes engines with different normal ranges, so it means little on its own. That is why the Genie instructions in Step 3 tell it to group by model before comparing EGT.

In [ ]:
display(
    spark.sql(f"""
        SELECT
            a.model,
            ROUND(AVG(r.value), 1) AS avg_egt,
            COUNT(*)               AS reading_count
        FROM {FQN}.sensor_readings r
        JOIN {FQN}.sensors  sen ON r.sensor_id = sen.sensor_id
        JOIN {FQN}.systems  s   ON sen.system_id = s.system_id
        JOIN {FQN}.aircraft a   ON s.aircraft_id = a.aircraft_id
        WHERE sen.type = 'EGT'
        GROUP BY a.model
        ORDER BY avg_egt DESC
    """)
)

## Step 2: Create the Genie Agent

Genie Agents are created in the Databricks UI, so the rest of this lab is click-through. Keep this notebook open beside the UI.

### 2.1 Navigate to Genie Agent

1. Click **Genie Agents** in the left sidebar, then **New**

### 2.2 Connect your data

The **Connect your data** dialog appears. Select **All Catalogs** > `databricks-neo4j-workshop` > `aircraft`, then select `sensor_readings`, `aircraft`, `sensors`, and `systems`.

> **Tip:** these tables form a join chain: `sensor_readings` -> `sensors` -> `systems` -> `aircraft`

![Connect your data dialog](https://raw.githubusercontent.com/neo4j-partners/databricks-neo4j-workshop/main/site/modules/ROOT/images/lab4-genie-connect-data.png)

> **Tip:** if you do not see the tables under **Recent**, click **All** or search for `databricks-neo4j-workshop`.

### 2.3 Configure basic settings

Once the agent exists, click **Configure** in the top navigation bar, then the **Settings** tab:

![Genie Agent Configure > Settings panel](https://raw.githubusercontent.com/neo4j-partners/databricks-neo4j-workshop/main/site/modules/ROOT/images/lab4-configure-genie.png)

1. **Name:** `Aircraft Sensor Analyst <YOUR INITIALS>`, for example `Aircraft Sensor Analyst RK`. Everyone in the room shares this workspace, so the initials are what tell your agent apart from thirty others.
2. **Description:** "Analyzes aircraft engine sensor telemetry including EGT, vibration, N1 speed, and fuel flow metrics"
3. **Common Questions:** Add the following:

> **The data stops on September 28, 2024.** Every question here names a real month rather than saying "recently" or "the last 30 days". Relative dates make Genie filter against today, which is far past the end of the data, and it returns zero rows.

**Time-Series Analytics**

What is the average EGT temperature for aircraft N10000 in September 2024?

**Fleet Comparisons**

Compare average EGT temperatures between Boeing 737 and Airbus A320 aircraft

**Anomaly Detection**

Find sensors with readings above their 95th percentile value

**Trend Analysis**

Show the trend of EGT temperatures over the 90-day period for aircraft N10000


## Step 3: Add Instructions

Go to **Configure** > **Instructions**. Instructions carry domain knowledge and query conventions: sensor types, normal ranges, and data conventions. Without them Genie writes plausible SQL rather than correct SQL.

Copy the whole block below into the **Instructions** field.

> **Two details in the block that look like typos and are not.** The `unit` column holds `C`, not `°C`, because the pipeline strips the degree sign so a generated `unit = 'C'` filter matches. And Vibration and N1Speed limits are maxima with no minimum, unlike EGT and FuelFlow, because that is how the maintenance manuals write them.

```
# Aircraft Sensor Analytics Domain Knowledge

## Sensor Types and Normal Ranges
Ranges come from each maintenance manual's takeoff limits, so they are per model. Always filter or group by model before comparing values across aircraft.

- EGT (Exhaust Gas Temperature): unit column value C, with no degree sign. Range per model: A320-200 620-680, A220-300 855-890, E190 870-900, B737-800 900-950, A321neo 980-1040.
- Vibration: unit column value ips. The limit is a maximum with no minimum, so a reading is only abnormal when it is above the limit for its model: A320-200 2.0, A220-300 2.5, A321neo 2.5, B737-800 3.0, E190 3.0.
- N1Speed (Fan Speed N1): unit column value % RPM. The limit is a maximum with no minimum: A321neo 97, A220-300 100, E190 100, A320-200 104, B737-800 104.
- FuelFlow: unit column value kg/s. Range per model: E190 1.00-1.20, A220-300 1.15-1.35, B737-800 1.20-1.50, A320-200 1.20-1.95, A321neo 1.50-2.00.

## Fleet Information
- Operators: ExampleAir, SkyWays, RegionalCo, NorthernJet
- Models: B737-800 by Boeing, A320-200 by Airbus, A321neo by Airbus, E190 by Embraer, A220-300 by Airbus

## Sensor Configuration
- Each aircraft has 2 engines
- Each engine has 4 sensors: EGT, Vibration, N1Speed, FuelFlow

## Data Conventions
- Timestamps are stored as timestamp type in the `timestamp` column
- Data period: July 1, 2024 to September 28, 2024 (90 days). There is no data outside it
- Never filter on CURRENT_DATE, NOW() or CURRENT_TIMESTAMP. Every one of those is years past the end of the data and return zero rows
- Read "recent", "the last month" and "the last 30 days" as September 2024
- Readings are every 4 hours (6 per day per sensor)

## Sensor ID Format
- Format: AC{aircraft_number}-S{system_number}-SN{sensor_number}
- Example: AC1001-S01-SN01 = Aircraft 1001, Engine 1 (S01), EGT sensor (SN01)
- S01 and S02 are always engines; S03 is Avionics; S04 is Hydraulics
- SN01=EGT, SN02=Vibration, SN03=N1Speed, SN04=FuelFlow

## Engine Names by Model
The `systems` table stores each engine as a system named after its engine model, so these are the exact strings to match on.
- B737-800: CFM56-7B
- A320-200: CFM56-5B
- A321neo: LEAP-1A
- E190: CF34-10E
- A220-300: PW1500G

## Query Conventions
- When asked about "Engine 1", filter by systems where name contains "#1"
- When asked about "Engine 2", filter by systems where name contains "#2"
- Use tail_number for human-readable aircraft references (e.g., N10000)
- Use aircraft_id for internal references (e.g., AC1001)
- Always report the unit alongside a value. Take it from the sensors.unit column, whose values are C, ips, % RPM and kg/s
```

## Step 4: Test the Genie Agent

### 4.1 Start a conversation

Click **Start conversation**, or open the chat interface.

### 4.2 Test basic queries

Work down this list. Each query is harder than the one before it.

**Query 1: Simple Aggregation**
```
What is the average EGT temperature across all sensors?
```
Expected: a single number around 865 degrees Celsius. The fleet mixes models whose EGT bands run from 620-680 on the A320-200 up to 980-1040 on the A321neo, so a fleet-wide average sits between them and is not meaningful on its own.

**Query 2: Filtering by Aircraft**
```
Show the average EGT for aircraft N10000
```
Expected: average EGT for that specific aircraft

**Query 3: Time-Series Trend**
```
Show daily average EGT for aircraft AC1001 in July 2024
```
Expected: about 30 rows with a date and an average value

**Query 4: Cross-Table Join**
```
Compare average vibration readings by aircraft model
```
Expected: one row per model, five in all: A220-300, A320-200, A321neo, B737-800, E190

**Query 5: Statistical Analysis**
```
Find the top 5 sensors with the highest average readings for their type
```
Expected: top sensors with their average values and types

### 4.3 Read the generated SQL

For each answer, click **View Code** and check the query Genie wrote.

Here is what a correct answer to "Compare average vibration by aircraft model" looks like:

```sql
SELECT
    a.model,
    AVG(r.value) as avg_vibration,
    COUNT(*) as reading_count
FROM sensor_readings r
JOIN sensors sen ON r.sensor_id = sen.sensor_id
JOIN systems s ON sen.system_id = s.system_id
JOIN aircraft a ON s.aircraft_id = a.aircraft_id
WHERE sen.type = 'Vibration'
GROUP BY a.model
ORDER BY avg_vibration DESC
```

Run the same query here and compare the numbers to what Genie returned.

In [ ]:
display(
    spark.sql(f"""
        SELECT
            a.model,
            AVG(r.value) AS avg_vibration,
            COUNT(*)     AS reading_count
        FROM {FQN}.sensor_readings r
        JOIN {FQN}.sensors  sen ON r.sensor_id = sen.sensor_id
        JOIN {FQN}.systems  s   ON sen.system_id = s.system_id
        JOIN {FQN}.aircraft a   ON s.aircraft_id = a.aircraft_id
        WHERE sen.type = 'Vibration'
        GROUP BY a.model
        ORDER BY avg_vibration DESC
    """)
)

## Step 5: Find the Agent ID

Lab 5 addresses this agent by its id, so record it now.

Open **About this agent** and copy the **Agent ID**. The same value is in the browser URL, after `/genie/rooms/`, if the panel is not where you expect it.

Paste it somewhere you will still have it in Lab 5. It is a 32-character hex string, for example `01f1661b55731a0293c3f84ac9c5ba52`.

## Summary

Your Genie Agent now:

- Answers questions about sensor telemetry in plain English
- Aggregates by aircraft, model, operator, or sensor type
- Runs statistical analysis: averages, percentiles, standard deviation
- Joins across the four tables to give context-rich answers
- Understands the domain vocabulary: EGT, N1Speed, and the rest

### More sample queries

Add these as sample questions, or use them to test further.

**Time-Series Analytics**

```
Show daily average vibration readings for Engine 1 on aircraft AC1001
```

```
What was the maximum fuel flow recorded in August 2024?
```

**Fleet Comparisons**

```
Which aircraft has the highest average vibration readings?
```

```
Show fuel flow rates by operator
```

**Anomaly Detection**

```
Show all EGT readings above 950 degrees Celsius for B737-800 aircraft
```

```
Which B737-800 engines have N1 speed readings above the 104 % RPM takeoff limit?
```

**Trend Analysis**

```
Calculate the 7-day rolling average of vibration for Engine 1 on AC1001
```

### Where Genie stops

Genie answers *how much* and *how often*. Average EGT on a tail number in September, the maximum fuel flow in August, which aircraft vibrates most. Every one of those is an aggregation over timestamped rows, and SQL over the Lakehouse is the right tool for all of them.

Two kinds of question it cannot reach, and better instructions do not fix either.

**Why a flight was delayed.** The gold `flights` table carries `total_delay_minutes`, so Genie can tell you a flight lost 40 minutes. What caused those minutes is a `Delay` node in Neo4j and exists nowhere in Unity Catalog. Genie hands you the number and stops.

**Traversals whose depth you do not know in advance.** Aircraft to system to component to maintenance event, then out to every other component carrying the same part number. In Cypher that is one variable-length pattern. In SQL it is a recursive query rewritten from scratch for each new question.

Note what is *not* on that list. `flights` and `maintenance_events` are published into this same `aircraft` schema, so Genie can reach those rows the moment you add the tables to it. The split is not "numbers in Delta, relationships in Neo4j". It is that a graph query language states a connected question in one line, and SQL does not.

There are two ways to give an agent both. You build one in code in Lab 5. The optional demo in
[`OPTIONAL_COMPOUND_AGENT_DEMO.md`](./OPTIONAL_COMPOUND_AGENT_DEMO.md) reaches the same place through
configuration instead, with a Neo4j MCP server behind a Unity Catalog connection.

---

## Next Steps

Continue to **Lab 5**. It builds a LangGraph supervisor in Python over three tools:

1. The Genie Agent you just created, addressed by the `GENIE_AGENT_ID` you recorded above
2. Cypher over **your own** Aura instance, the one you loaded in Lab 2
3. The GraphRAG retrievers you built in Lab 3

It ends with the agent deployed to Model Serving and authenticating as a service principal.

**Before you leave this notebook:** check that you recorded the `GENIE_AGENT_ID`. Lab 5 starts by asking for it.

After the workshop:
- Add a subagent for maintenance procedure search
- Build custom Unity Catalog functions as extra tools
- Add guardrails and output validation
- Give the agent memory in Neo4j, which is **Lab 6**